# Lezione 7 — Apprendimento per rinforzo: insegna a un'IA a giocare

**Obiettivi di oggi**
- Descrivere come un agente impara per prova ed errore con ricompense.
- Generare dati giocando partite casuali con OpenAI Gym.
- Addestrare una rete che gioca a CartPole meglio del caso.

Consulta il [capitolo 6 del manuale](../manual/06_reinforcement_learning.md) per la spiegazione completa.

In [ ]:
!pip install -q gym

## 1. Esplora l'ambiente CartPole

In [ ]:
import gym
import numpy as np
import tensorflow as tf

entorno = gym.make("CartPole-v1")
observacion, info = entorno.reset()

print("Forma de la observación:", observacion.shape)
print("Acciones posibles:", entorno.action_space.n)

## 2. Baseline: quanto dura un agente che agisce a caso?

In [ ]:
def jugar_episodio_aleatorio(entorno):
    observacion, info = entorno.reset()
    pasos = 0
    terminado = False
    while not terminado:
        accion = entorno.action_space.sample()
        observacion, recompensa, terminado, truncado, info = entorno.step(accion)
        pasos += 1
        if truncado:
            break
    return pasos

pasos_aleatorios = [jugar_episodio_aleatorio(entorno) for _ in range(20)]
print("Pasos medios con acciones al azar:", np.mean(pasos_aleatorios))

## 3. Definisci la rete che stima il valore di ogni azione (DQN)

In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(24, activation="relu"),
    tf.keras.layers.Dense(24, activation="relu"),
    tf.keras.layers.Dense(2, activation="linear"),
])
modelo.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss="mse")

def elegir_accion(observacion, epsilon):
    if np.random.random() < epsilon:
        return np.random.randint(2)
    valores_q = modelo.predict(observacion.reshape(1, -1), verbose=0)
    return int(np.argmax(valores_q[0]))

## 4. Addestra l'agente con experience replay

Salviamo le esperienze (stato, azione, ricompensa, stato successivo) in un buffer, e addestriamo con lotti casuali da quel buffer per stabilizzare l'apprendimento.

In [ ]:
from collections import deque
import random

buffer = deque(maxlen=2000)
epsilon = 1.0
gamma = 0.95

for episodio in range(30):
    observacion, info = entorno.reset()
    recompensa_total = 0
    terminado = False
    while not terminado:
        accion = elegir_accion(observacion, epsilon)
        siguiente_obs, recompensa, terminado, truncado, info = entorno.step(accion)
        buffer.append((observacion, accion, recompensa, siguiente_obs, terminado))
        observacion = siguiente_obs
        recompensa_total += recompensa
        if truncado:
            break

        if len(buffer) >= 32:
            lote = random.sample(buffer, 32)
            estados = np.array([e[0] for e in lote])
            acciones = np.array([e[1] for e in lote])
            recompensas = np.array([e[2] for e in lote])
            siguientes = np.array([e[3] for e in lote])
            terminados = np.array([e[4] for e in lote])

            objetivos_q = modelo.predict(estados, verbose=0)
            valores_futuros = np.max(modelo.predict(siguientes, verbose=0), axis=1)
            for i in range(32):
                objetivos_q[i, acciones[i]] = recompensas[i] + (0 if terminados[i] else gamma * valores_futuros[i])
            modelo.fit(estados, objetivos_q, verbose=0)

    epsilon = max(0.05, epsilon * 0.95)
    if episodio % 5 == 0:
        print(f"Episodio {episodio}: recompensa {recompensa_total}, epsilon {epsilon:.2f}")

## Riepilogo

- Un agente con azioni casuali dura pochi step in CartPole.
- Un DQN impara, sulla base di prove ed errori registrati in un buffer di esperienze, a stimare quale azione è conveniente in ogni stato.
- `epsilon` decrescente fa sì che l'agente esplori molto all'inizio e sfrutti sempre più la sua conoscenza man mano che impara.

Continua con il [notebook de práctica di questa lezione](../notebooks_practica/clase07_practica_rl_cartpole.ipynb).